In [0]:
import time
import urllib.request

CATALOGO   = "nyc_taxi_andres"
CAPA_RAW   = f"{CATALOGO}.raw"

URL_VIAJES = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
URL_ZONAS  = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

VOLUMEN    = "/Volumes/nyc_taxi_andres/raw/archivos_fuente"
RUTA_VIAJES = f"{VOLUMEN}/viajes_enero_2023.parquet"
RUTA_ZONAS  = f"{VOLUMEN}/zonas_taxi.csv"

print("Configuracion lista")

In [0]:
# descargamos los archivos al volumen de Unity Catalog
# desde ahi Spark puede leer los archivos

print("Descargando archivos al volumen...")
inicio = time.time()

urllib.request.urlretrieve(URL_VIAJES, RUTA_VIAJES)
print("Viajes descargados")

urllib.request.urlretrieve(URL_ZONAS, RUTA_ZONAS)
print("Zonas descargadas")

print(f"Descarga lista en {round(time.time() - inicio, 2)}s")

In [0]:
# leemos viajes con Spark desde el volumen
print("Leyendo viajes con Spark...")
inicio = time.time()

viajes_crudos = spark.read.parquet(RUTA_VIAJES)

total_viajes = viajes_crudos.count()
print(f"Viajes leidos: {total_viajes:,} en {round(time.time() - inicio, 2)}s")

viajes_crudos.printSchema()

In [0]:
# leemos las zonas con Spark
zonas = spark.read.csv(RUTA_ZONAS, header=True, inferSchema=True)

print(f"Zonas leidas: {zonas.count()}")
zonas.show(5)

In [0]:
# guardamos todo en Delta dentro del schema raw
# Delta nos da trazabilidad y la posibilidad de volver
# atrás si algo sale mal en las capas siguientes

print("Guardando en la capa Raw del catálogo...")
inicio = time.time()

try:
    viajes_crudos.write.format("delta").mode("overwrite").saveAsTable(f"{CAPA_RAW}.viajes_enero_2023")
    zonas.write.format("delta").mode("overwrite").saveAsTable(f"{CAPA_RAW}.zonas_taxi")
    print(f"Tablas guardadas en {round(time.time() - inicio, 2)}s")
except Exception as e:
    print(f"Error guardando en Raw: {e}")
    raise

In [0]:
# verificamos que las tablas quedaron bien antes de seguir
viajes_guardados = spark.table(f"{CAPA_RAW}.viajes_enero_2023").count()
zonas_guardadas  = spark.table(f"{CAPA_RAW}.zonas_taxi").count()

print("  RESUMEN — CAPA RAW")
print("=" * 50)
print(f"  Viajes guardados : {viajes_guardados:,}")
print(f"  Zonas guardadas  : {zonas_guardadas}")
print("  Siguiente paso   : limpieza en Trusted")
